In [0]:
import json
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import RunResultState

In [0]:
COMMIT_ID = "31fcded52b5a0e586210071bdf28fd1ab54a5f38"
COMMIT_MSG = "Update Synthetic_Data_CMDB.ipynb"


In [0]:
if not COMMIT_ID or not COMMIT_MSG:
    raise ValueError("COMMIT_ID and COMMIT_MSG must be defined")

w = WorkspaceClient()

def get_job_id(name):
    return next(j.job_id for j in w.jobs.list() if j.settings.name == name)

JOB_ID = get_job_id("synthetic_data_bank_run_on_merge")
TRIGGER_JOB_ID = get_job_id("synthetic_data_bank")

TASK_KEY = "Check_Commit_Info"

# -------------------------------------------------
# STATUS TRACKING
# -------------------------------------------------
status = "UNKNOWN"
downstream_triggered = False

# -------------------------------------------------
# FIND LAST SUCCESSFUL JOB RUN
# -------------------------------------------------
successful_run = None

for run in w.jobs.list_runs(job_id=JOB_ID, limit=10):
    if run.state and run.state.result_state == RunResultState.SUCCESS:
        successful_run = run
        break

if not successful_run:
    status = "NO_PREVIOUS_RUN"

else:
    full_run = w.jobs.get_run(run_id=successful_run.run_id)

    task_run = next(
        (t for t in (full_run.tasks or []) if t.task_key == TASK_KEY),
        None
    )

    if not task_run:
        status = "TASK_NOT_FOUND"

    elif task_run.state.result_state != RunResultState.SUCCESS:
        status = "TASK_NOT_SUCCESSFUL"

    else:
        output = w.jobs.get_run_output(run_id=task_run.run_id)

        payload_raw = (
            output.notebook_output.result
            if output.notebook_output and output.notebook_output.result
            else None
        )

        if not payload_raw:
            status = "NO_PREVIOUS_PAYLOAD"

        else:
            try:
                payload = json.loads(payload_raw)

                prev_commit_id = payload.get("commit_id")
                prev_commit_msg = payload.get("commit_msg")

                if (
                    prev_commit_id == COMMIT_ID
                    and prev_commit_msg == COMMIT_MSG
                ):
                    status = "DUPLICATE_COMMIT"
                    downstream_triggered = False

                    dbutils.notebook.exit(
                        json.dumps({
                            "status": status,
                            "downstream_triggered": downstream_triggered,
                            "commit_id": COMMIT_ID,
                            "commit_msg": COMMIT_MSG
                        })
                    )

                status = "NEW_COMMIT"

            except json.JSONDecodeError:
                status = "INVALID_PREVIOUS_PAYLOAD"

# -------------------------------------------------
# TRIGGER DOWNSTREAM
# -------------------------------------------------
downstream_triggered = True
w.jobs.run_now(job_id=TRIGGER_JOB_ID)
# -------------------------------------------------
# RETURN COMMIT INFO TO JOB CONTROLLER ✅
# -------------------------------------------------
final_payload = {
    "status": status,
    "downstream_triggered": downstream_triggered,
    "commit_id": COMMIT_ID,
    "commit_msg": COMMIT_MSG
}

dbutils.notebook.exit(json.dumps(final_payload))